In [1]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder
import pytorch_lightning as pl

class CryptoDataset(Dataset):
    def __init__(self, features: pd.DataFrame, labels: pd.DataFrame):
        """
        Args:
            features (pd.DataFrame): DataFrame containing the preprocessed feature data.
            labels (pd.DataFrame): DataFrame containing the unprocessed labels.
        """
        self.features = features.values  # Convert features to NumPy array
        self.labels = labels['&-target']  # Extract the labels column

        # Encode the labels (e.g., 'up'/'down') into integers
        self.label_encoder = LabelEncoder()
        self.labels_encoded = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        # Return the total number of samples in the dataset
        return len(self.features)

    def __getitem__(self, idx):
        """
        Returns a single sample from the dataset, indexed by `idx`.
        """
        feature = torch.tensor(self.features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels_encoded[idx], dtype=torch.long)
        return feature, label

class CryptoDataModule(pl.LightningDataModule):
    def __init__(self, directory_path, batch_size=64, train_split=0.9):
        super().__init__()
        self.directory_path = directory_path
        self.batch_size = batch_size
        self.train_split = train_split
        self.input_dim = None
        self.output_dim = None

    def setup(self, stage=None):
        """
        Load data and split into training and validation datasets.
        Automatically infer input_dim and output_dim.
        """
        features_path = os.path.join(self.directory_path, 'features_filtered.parquet')
        labels_path = os.path.join(self.directory_path, 'labels_filtered.parquet')

        print("Loading parquet files...")
        df_features = pd.read_parquet(features_path)
        df_labels = pd.read_parquet(labels_path)
        print("Parquet files loaded successfully.")

        # Infer input_dim (number of features)
        self.input_dim = df_features.shape[1]  # Number of columns in the features DataFrame

        # Infer output_dim (number of classes)
        label_encoder = LabelEncoder()
        self.output_dim = len(label_encoder.fit(df_labels['&-target']).classes_)

        # Create dataset
        dataset = CryptoDataset(features=df_features, labels=df_labels)

        # Split dataset into training and validation sets
        train_size = int(self.train_split * len(dataset))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

# Main execution
if __name__ == "__main__":
    # Directory path
    directory_path = '/allah/data/parquet'
    os.chdir(directory_path)
    # Initialize DataModule
    data_module = CryptoDataModule(directory_path=directory_path, batch_size=64)

    # Setup datasets
    data_module.setup()

    input_dim = data_module.input_dim  # Automatically inferred from dataset
    output_dim = data_module.output_dim  # Automatically inferred from dataset

    # Dynamically set hidden_dim based on input_dim (optional)
    hidden_dim = max(32, input_dim // 2)  # Example heuristic: at least 32 units, or half of input_dim


    train_loader = data_module.train_dataloader()
    
    for batch_features, batch_labels in train_loader:
        print(f"Features batch shape: {batch_features.shape}")
        print(f"Labels batch shape: {batch_labels.shape}")
        break

    # Test validation DataLoader
    val_loader = data_module.val_dataloader()
    for batch_features, batch_labels in val_loader:
        print(f"Validation Features batch shape: {batch_features.shape}")
        print(f"Validation Labels batch shape: {batch_labels.shape}")
        break


Loading parquet files...
Parquet files loaded successfully.
Features batch shape: torch.Size([64, 626])
Labels batch shape: torch.Size([64])
Validation Features batch shape: torch.Size([64, 626])
Validation Labels batch shape: torch.Size([64])


In [2]:
import pandas as pd
import os
import torch

# Set the directory path
directory_path = '/allah/data/parquet'
os.chdir(directory_path)

# Load the features and labels parquet files
df = pd.read_parquet('features_filtered.parquet')
df_labeld = pd.read_parquet('labels_filtered.parquet')

# Display the head of the features dataframe
print("Features Dataframe:")
print(df.head())

# Check label distribution
print("\nLabel Distribution:")
print(df_labeld['&-target'].value_counts())

# If using a DataModule (e.g., PyTorch Lightning), load train and validation data
# Replace 'data_module' with your actual DataModule instance
train_loader = data_module.train_dataloader()
val_loader = data_module.val_dataloader()

# Extract all labels from the training and validation datasets
train_labels = torch.cat([batch_labels for _, batch_labels in train_loader])
val_labels = torch.cat([batch_labels for _, batch_labels in val_loader])

# Calculate label distribution for training and validation sets
train_label_counts = torch.bincount(train_labels)
val_label_counts = torch.bincount(val_labels)

# Print training label distribution
print("\nTraining Label Distribution:")
for i, count in enumerate(train_label_counts):
    print(f"Class {i}: {count.item()} samples")

# Print validation label distribution
print("\nValidation Label Distribution:")
for i, count in enumerate(val_label_counts):
    print(f"Class {i}: {count.item()} samples")


Features Dataframe:
   %-rsi-period_10_ETH/USDTUSDT_3m  %-mfi-period_10_ETH/USDTUSDT_3m  \
0                        -0.019943                        -0.193720   
1                         0.208868                        -0.183873   
2                         0.074703                        -0.075862   
3                         0.197583                         0.187643   
4                        -0.268256                        -0.675241   

   %-adx-period_10_ETH/USDTUSDT_3m  %-sma-period_10_ETH/USDTUSDT_3m  \
0                         0.329148                        -0.866667   
1                         0.269632                        -0.899361   
2                         0.195698                        -0.899326   
3                         0.136397                        -0.898825   
4                        -0.641939                        -0.892534   

   %-ema-period_10_ETH/USDTUSDT_3m  %-bb_width-period_10_ETH/USDTUSDT_3m  \
0                        -0.869929                

In [3]:
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl

class CryptoPricePredictor(pl.LightningModule):
    def __init__(self, input_dim, hidden_dim=32, output_dim=3, dropout_rate=0.1):  # Updated output_dim for multiple classes
        super(CryptoPricePredictor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # First hidden layer
            nn.BatchNorm1d(hidden_dim),         # Batch normalization for better training stability
            nn.ReLU(),                          # Activation function
            nn.Dropout(dropout_rate),           # Dropout for regularization

            nn.Linear(hidden_dim, hidden_dim),  # Second hidden layer
            nn.BatchNorm1d(hidden_dim),         # Batch normalization
            nn.ReLU(),                          # Activation function
            nn.Dropout(dropout_rate),           # Dropout

            nn.Linear(hidden_dim, output_dim)   # Output layer
        )
        self.output_dim = output_dim  # Store the number of classes

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        features, labels = batch
        outputs = self(features)
        loss = F.cross_entropy(outputs, labels)
        
        # Training Accuracy
        acc = (outputs.argmax(dim=1) == labels).float().mean()
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_acc', acc, on_step=True, on_epoch=True, prog_bar=True)
        
        # Precision for each class
        preds = outputs.argmax(dim=1)  # Get predicted classes
        precisions = {}  # Dictionary to store precision for each class

        for cls in range(self.output_dim):
            tp = ((preds == cls) & (labels == cls)).float().sum()  # True Positives for class `cls`
            fp = ((preds == cls) & (labels != cls)).float().sum()  # False Positives for class `cls`
            precision = tp / (tp + fp + 1e-8)  # Precision for class `cls`
            precisions[f'train_precision_class_{cls}'] = precision
            self.log(f'train_precision_class_{cls}', precision, on_step=True, on_epoch=True, prog_bar=True)
        
        return loss

    def validation_step(self, batch, batch_idx):
        features, labels = batch
        outputs = self(features)
        loss = F.cross_entropy(outputs, labels)
        
        # Validation Accuracy
        acc = (outputs.argmax(dim=1) == labels).float().mean()
        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('val_acc', acc, on_step=True, on_epoch=True, prog_bar=True)
        
        # Precision for each class
        preds = outputs.argmax(dim=1)  # Get predicted classes
        precisions = {}  # Dictionary to store precision for each class

        for cls in range(self.output_dim):
            tp = ((preds == cls) & (labels == cls)).float().sum()  # True Positives for class `cls`
            fp = ((preds == cls) & (labels != cls)).float().sum()  # False Positives for class `cls`
            precision = tp / (tp + fp + 1e-8)  # Precision for class `cls`
            precisions[f'val_precision_class_{cls}'] = precision
            self.log(f'val_precision_class_{cls}', precision, on_step=True, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl

class CryptoPricePredictor(pl.LightningModule):
    def __init__(self, input_dim, hidden_dim=32, dropout_rate=0.1):
        super(CryptoPricePredictor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # First hidden layer
            nn.BatchNorm1d(hidden_dim),         # Batch normalization
            nn.ReLU(),                          # Activation function
            nn.Dropout(dropout_rate),           # Dropout for regularization

            nn.Linear(hidden_dim, hidden_dim),  # Second hidden layer
            nn.BatchNorm1d(hidden_dim),         # Batch normalization
            nn.ReLU(),                          # Activation function
            nn.Dropout(dropout_rate),           # Dropout for regularization

            nn.Linear(hidden_dim, 1)            # Output layer for binary classification
        )

    def forward(self, x):
        return self.model(x)  # Sigmoid will be applied in the loss function for numerical stability

    def training_step(self, batch, batch_idx):
        features, labels = batch
        outputs = self(features).squeeze()  # Ensure outputs shape matches labels (N,)

        # Binary Cross-Entropy Loss
        loss = F.binary_cross_entropy_with_logits(outputs, labels.float())

        # Binary Predictions (Threshold = 0.5)
        preds = (torch.sigmoid(outputs) > 0.5).long()

        # Training Accuracy
        acc = (preds == labels).float().mean()
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_acc', acc, on_step=True, on_epoch=True, prog_bar=True)

        # Precision for each class
        precisions = {}
        for cls in [0, 1]:  # Binary classes
            tp = ((preds == cls) & (labels == cls)).float().sum()  # True Positives
            fp = ((preds == cls) & (labels != cls)).float().sum()  # False Positives
            precision = tp / (tp + fp + 1e-8)  # Avoid division by zero
            precisions[f'train_precision_class_{cls}'] = precision
            self.log(f'train_precision_class_{cls}', precision, on_step=True, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        features, labels = batch
        outputs = self(features).squeeze()

        # Binary Cross-Entropy Loss
        loss = F.binary_cross_entropy_with_logits(outputs, labels.float())

        # Binary Predictions (Threshold = 0.5)
        preds = (torch.sigmoid(outputs) > 0.5).long()

        # Validation Accuracy
        acc = (preds == labels).float().mean()
        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('val_acc', acc, on_step=True, on_epoch=True, prog_bar=True)

        # Precision for each class
        precisions = {}
        for cls in [0, 1]:  # Binary classes
            tp = ((preds == cls) & (labels == cls)).float().sum()  # True Positives
            fp = ((preds == cls) & (labels != cls)).float().sum()  # False Positives
            precision = tp / (tp + fp + 1e-8)  # Avoid division by zero
            precisions[f'val_precision_class_{cls}'] = precision
            self.log(f'val_precision_class_{cls}', precision, on_step=True, on_epoch=True, prog_bar=True)

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)


In [6]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.tensorboard import SummaryWriter


# Define the logger (optional)
logger = TensorBoardLogger(
    save_dir='/allah/data/parquet')

# Define input dimensions based on the features

# Initialize the model
model = CryptoPricePredictor(input_dim=input_dim, hidden_dim=hidden_dim)

# Initialize the data module
directory_path = '/allah/data/parquet'  # Update with your actual directory path
data_module = CryptoDataModule(directory_path=directory_path, batch_size=64)

# Initialize EarlyStopping callback
early_stopping = EarlyStopping(
    monitor="val_loss",  # Metric to monitor
    mode="min",          # "min" for minimizing (e.g., loss), "max" for maximizing (e.g., accuracy)
    patience=10,          # Number of epochs with no improvement before stopping
    verbose=True         # Print messages when stopping
)

# Initialize the Trainer with the EarlyStopping callback
trainer = Trainer(max_epochs=1000, callbacks=[early_stopping], logger=logger)

# Train the model
trainer.fit(model, datamodule=data_module)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



  | Name  | Type       | Params | Mode 
---------------------------------------------
0 | model | Sequential | 296 K  | train
---------------------------------------------
296 K     Trainable params
0         Non-trainable params
296 K     Total params
1.184     Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode


Loading parquet files...
Parquet files loaded successfully.
                                                                            

/allah/freqtrade/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/allah/freqtrade/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 145/145 [00:02<00:00, 62.71it/s, v_num=0, train_loss_step=0.685, train_acc_step=0.614, train_precision_class_0_step=0.636, train_precision_class_1_step=0.591, val_loss_step=0.594, val_acc_step=0.600, val_precision_class_0_step=0.333, val_precision_class_1_step=1.000, val_loss_epoch=0.672, val_acc_epoch=0.586, val_precision_class_0_epoch=0.615, val_precision_class_1_epoch=0.565, train_loss_epoch=0.698, train_acc_epoch=0.532, train_precision_class_0_epoch=0.540, train_precision_class_1_epoch=0.524]

...

Epoch 31: 100%|██████████| 145/145 [00:02<00:00, 61.38it/s, v_num=0, train_loss_step=0.593, train_acc_step=0.750, train_precision_class_0_step=0.792, train_precision_class_1_step=0.700, val_loss_step=0.537, val_acc_step=0.600, val_precision_class_0_step=0.000, val_precision_class_1_step=0.750, val_loss_epoch=0.506, val_acc_epoch=0.760, val_precision_class_0_epoch=0.767, val_precision_class_1_epoch=0.752, train_loss_epoch=0.378, train_acc_epoch=0.831, train_precision_class_0_epoch=0.837, train_precision_class_1_epoch=0.826]

Metric val_loss improved by 0.011 >= min_delta = 0.0. New best score: 0.506


Epoch 41: 100%|██████████| 145/145 [00:02<00:00, 61.74it/s, v_num=0, train_loss_step=0.362, train_acc_step=0.773, train_precision_class_0_step=0.810, train_precision_class_1_step=0.739, val_loss_step=0.512, val_acc_step=0.800, val_precision_class_0_step=0.000, val_precision_class_1_step=0.800, val_loss_epoch=0.516, val_acc_epoch=0.766, val_precision_class_0_epoch=0.763, val_precision_class_1_epoch=0.766, train_loss_epoch=0.316, train_acc_epoch=0.861, train_precision_class_0_epoch=0.865, train_precision_class_1_epoch=0.857]

Monitored metric val_loss did not improve in the last 10 records. Best score: 0.506. Signaling Trainer to stop.


Epoch 41: 100%|██████████| 145/145 [00:02<00:00, 60.83it/s, v_num=0, train_loss_step=0.362, train_acc_step=0.773, train_precision_class_0_step=0.810, train_precision_class_1_step=0.739, val_loss_step=0.512, val_acc_step=0.800, val_precision_class_0_step=0.000, val_precision_class_1_step=0.800, val_loss_epoch=0.516, val_acc_epoch=0.766, val_precision_class_0_epoch=0.763, val_precision_class_1_epoch=0.766, train_loss_epoch=0.316, train_acc_epoch=0.861, train_precision_class_0_epoch=0.865, train_precision_class_1_epoch=0.857]


In [ ]:
!ls /allah/stuff/freq/userdir/backtest_results